# Testing an Encryption Algorithm

We are going to focus on implementing the [Caesar cipher](https://en.wikipedia.org/wiki/Caesar_cipher).

This simple encryption method involves rotating characters in the alphabet (and does not affect other characters like punctuation or spaces).

## Setting up the Working Environment

Run the following cell **twice** (the first run will install the dependencies and restart the notebook) to retrieve the required material and place it in the `/content/caesar-cipher/` folder. This command will also change the notebook's current directory to point to this folder.

In [ ]:
import pathlib

%set_env UV_PROJECT_ENVIRONMENT /usr/local

root_dir = pathlib.Path("/content") / "caesar-cipher"

if pathlib.Path.cwd() != root_dir and root_dir.exists():
  %cd {root_dir}


try:
  import caesar_cipher.caesar_cipher
except ImportError:
  !pip install -U uv
  if not root_dir.exists():
    !git clone https://github.com/shuuchuu/caesar-cipher.git {root_dir}
  %cd {root_dir}
  import os

  !uv sync --inexact
  os.kill(os.getpid(), 9)

## Tests from the Specification

Without looking at the `src/caesar_cipher/caesar_cipher.py` file, and based on the description of the Caesar cipher on Wikipedia (especially the examples given), write some simple tests (as functions) that check the encryption of a string by the provided `rotate` function, then run them.

In [ ]:
from caesar_cipher.caesar_cipher import rotate


# Your code here

## TDD Approach (Test Driven Development)

We want to add a function `crack(ciphertext, known_word)` that recovers the key of an encrypted message when one of its words is known. Develop it following the TDD cycle:

1. Write a test that checks that `crack(rotate("attaque à l'aube", 3), "aube")` is `3`, and run it: it must fail, since `crack` does not exist yet.
2. Write the minimal code that makes it pass.
3. Add a test that checks that `crack` raises a `ValueError` when no key reveals the word, see it fail, then make it pass.
4. Refactor your code if needed, checking that the tests still pass.

In [ ]:
# Your code here

### Solution

In [ ]:
def test_crack_finds_key():
    assert crack(rotate("attaque à l'aube", 3), "aube") == 3


def test_crack_without_key():
    try:
        crack("bonjour", "xyz")
    except ValueError:
        pass
    else:
        raise AssertionError("ValueError expected")


def crack(ciphertext: str, known_word: str) -> int:
    for key in range(26):
        if known_word in rotate(ciphertext, -key):
            return key
    raise ValueError(f"No key reveals {known_word!r}")


test_crack_finds_key()
test_crack_without_key()

## Using `pytest`

Create a `tests/test_caesar_cipher.py` file and adapt the functions that test `rotate` to make them compatible with `pytest`, grouping the examples in a single test with `@pytest.mark.parametrize`.

Then use `pytest` to run the tests.

Finally, complete your test file to achieve 5 test cases.

In [ ]:
!python -m pytest

## Mocking with `unittest.mock`

The `encrypt_with_random_key` function below encrypts a text with a randomly drawn key: its result changes on every call, which makes it hard to test.

Write a test that replaces `random.randint` with a mock that always returns `3`, with [`unittest.mock.patch`](https://docs.python.org/3/library/unittest.mock.html#unittest.mock.patch), then checks the returned encrypted text and key, and that `random.randint` was called once, with the arguments `(1, 25)`.

In [ ]:
import random

from caesar_cipher.caesar_cipher import rotate


def encrypt_with_random_key(text: str) -> tuple[str, int]:
    key = random.randint(1, 25)
    return rotate(text, key), key

In [ ]:
# Your code here

### Solution

In [ ]:
from unittest import mock


def test_encrypt_with_random_key():
    with mock.patch("random.randint", return_value=3) as randint:
        assert encrypt_with_random_key("abc") == ("def", 3)
    randint.assert_called_once_with(1, 25)


test_encrypt_with_random_key()

## Going Further

### Property-based Testing

Use [`hypothesis`](https://hypothesis.readthedocs.io/en/latest/) to test the `rotate` function with random data.

### Contracts

Finally, integrate a pre-condition contract into the `src/caesar_cipher/caesar_cipher.py` code. Use the [`deal`](https://deal.readthedocs.io/) library for this purpose.

### Mutation Testing

Use [`mutmut`](https://mutmut.readthedocs.io/) to find the behaviors your tests do not check: each mutation that passes your test suite reveals one.

Iterate to reduce the number of mutations that pass your test suite.

In [ ]:
!mutmut run

In [ ]:
!mutmut results

## Solution

The solutions of the exercises that create test files are on the repository's [`solution` branch](https://github.com/shuuchuu/caesar-cipher/tree/solution).